# Example 02 — Ingest a folder of PDFs

Demonstrates idempotent ingest: rerunning the same files is a no-op.

In [ ]:
from pathlib import Path

import httpx

BASE = 'http://localhost:8000'
LIBRARY = 'demo'
client = httpx.Client(base_url=BASE, timeout=600)

## Upload every PDF in a directory

In [ ]:
papers = Path('~/papers/rag-agent').expanduser()
for pdf in papers.glob('*.pdf'):
    with pdf.open('rb') as fp:
        res = client.post(
            f'/v1/libraries/{LIBRARY}/ingest',
            files={'file': (pdf.name, fp, 'application/pdf')},
        )
    payload = res.json()
    print(f"{res.status_code}  {pdf.name:40}  doc_id={payload.get('doc_id')}  chunks={payload.get('chunks_upserted')}")

## Re-running is a no-op (idempotent)

Same files, same hashes → same `IngestResponse`, no parse/embed/upsert.

In [ ]:
for pdf in papers.glob('*.pdf'):
    with pdf.open('rb') as fp:
        res = client.post(
            f'/v1/libraries/{LIBRARY}/ingest',
            files={'file': (pdf.name, fp, 'application/pdf')},
        )
    print(res.json())

## Inspect Library stats

In [ ]:
print(client.get(f'/v1/libraries/{LIBRARY}/stats').json())